# baseline_v8_finetune — local LoRA on Kaggle
Planner → caller → summarizer; three independent adapters, no global stage.

Default `PROFILE="smoke"` runs 20 episodes and two optimizer updates per role, including caller/summarizer negatives. Use `PROFILE="train"` for the full data run. Each role runs on one GPU in a fresh process. Console logs, runtime versions, finite weight-update checks, validation metrics and adapters are saved to the output folder. This notebook never publishes models or uploads training data to GitHub.

Training uses an isolated Python environment with torch 2.9.1 / CUDA 12.8. No global PyTorch replacement. Each role checks attention forward/backward before downloading/loading Qwen.


In [ ]:
from pathlib import Path
import subprocess
import sys
import json

REPO_URL = "https://github.com/tnisl/baseline.git"
GIT_REF = "v8_finetune"  # Set an exact commit SHA to reproduce a saved run
SOURCE_DIR = Path("/tmp/baseline_v8_finetune_source")
TRAIN_ENV = Path("/tmp/baseline_v8_finetune_env")
TRAIN_PYTHON = TRAIN_ENV / "bin/python"
PROFILE = "smoke"  # "smoke" or "train"
DATA_ROOT = None  # auto-discover attached train/train.jsonl
WORK_DIR = Path("/kaggle/working/baseline_v8_finetune") / PROFILE
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
GPU_INDEX = 0  # select one GPU even if the session has T4 x2
CUDA_DEBUG = False  # True only to locate an asynchronous CUDA failure
EXTRA_ARGS = []  # e.g. ["--epochs", "2"] for a new train experiment

if not SOURCE_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(SOURCE_DIR)], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", SOURCE_COMMIT, flush=True)
print("Profile:", PROFILE, "Output:", WORK_DIR, flush=True)


In [ ]:
# Isolate training from Kaggle's global torch/vision/inference packages.
subprocess.run([sys.executable, str(SOURCE_DIR / "setup_training_env.py"),
                "--venv-dir", str(TRAIN_ENV)], check=True, timeout=1800)


In [ ]:
command = [str(TRAIN_PYTHON), "-u", str(SOURCE_DIR / "finetune.py"),
           "--profile", PROFILE, "--output-dir", str(WORK_DIR),
           "--model", MODEL_ID, "--gpu-index", str(GPU_INDEX)]
if DATA_ROOT:
    command += ["--data-root", str(DATA_ROOT)]
if CUDA_DEBUG:
    command.append("--cuda-debug")
command += list(EXTRA_ARGS)
print("Run:", " ".join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
status = json.loads((WORK_DIR / "run_status.json").read_text())
assert status["status"] == "complete", status
assert set(status["completed_roles"]) == {"planner", "caller", "summarizer"}, status
print(json.dumps(status, ensure_ascii=False, indent=2))
metrics = json.loads((WORK_DIR / "metrics.json").read_text())
for role, record in metrics.items():
    assert record["adapter_verification"]["finite"]
    assert record["adapter_verification"]["updated"]
    print(role, json.dumps({"validation": record["validation"],
                            "verification": record["adapter_verification"],
                            "wall_s": record["wall_s"]}, ensure_ascii=False))
print("Adapter ZIP:", WORK_DIR / "lora_adapters.zip")
